In [0]:
# Cell 1 — Read directly with credentials in path
storage_account_name = "populationpipelinesa"
storage_account_key = "YOUR_STORAGE_ACCOUNT_KEY_HERE"

file_path = "abfss://bronze@populationpipelinesa.dfs.core.windows.net/Top 100 Worlds Largest Cities.csv"

df_bronze = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("fs.azure.account.key.populationpipelinesa.dfs.core.windows.net", storage_account_key)
    .csv(file_path))

print("Rows:", df_bronze.count())
df_bronze.show(5)

Rows: 100
+----+---------+----------+-----------------+------------+
|Rank|     City|   Country|Population (Est.)|Area (sq km)|
+----+---------+----------+-----------------+------------+
|   1|    Tokyo|     Japan|       37,115,000|       8,231|
|   2|    Delhi|     India|       35,500,000|       2,344|
|   3| Shanghai|     China|       31,050,000|       4,333|
|   4|    Dhaka|Bangladesh|       25,360,000|       2,570|
|   5|Sao Paulo|    Brazil|       23,170,000|       3,649|
+----+---------+----------+-----------------+------------+
only showing top 5 rows


In [0]:
# Cell 2 - Clean the data
from pyspark.sql.functions import col, regexp_replace, trim

df_silver = df_bronze \
    .withColumnRenamed("Population (Est.)", "population") \
    .withColumnRenamed("Area (sq km)", "area_sq_km") \
    .withColumn("City", trim(col("City"))) \
    .withColumn("Country", trim(col("Country"))) \
    .withColumn("population", regexp_replace(col("population").cast("string"), ",", "").cast("long")) \
    .withColumn("area_sq_km", regexp_replace(col("area_sq_km").cast("string"), ",", "").cast("double")) \
    .dropna()

print("Cleaned Rows:", df_silver.count())
df_silver.show(5)

Cleaned Rows: 100
+----+---------+----------+----------+----------+
|Rank|     City|   Country|population|area_sq_km|
+----+---------+----------+----------+----------+
|   1|    Tokyo|     Japan|  37115000|    8231.0|
|   2|    Delhi|     India|  35500000|    2344.0|
|   3| Shanghai|     China|  31050000|    4333.0|
|   4|    Dhaka|Bangladesh|  25360000|    2570.0|
|   5|Sao Paulo|    Brazil|  23170000|    3649.0|
+----+---------+----------+----------+----------+
only showing top 5 rows


In [0]:
# Cell 3 - Save to Silver container
silver_path = "abfss://silver@populationpipelinesa.dfs.core.windows.net/cities_cleaned"

df_silver.write \
    .mode("overwrite") \
    .option("fs.azure.account.key.populationpipelinesa.dfs.core.windows.net", storage_account_key) \
    .parquet(silver_path)

print("✅ Silver layer saved successfully!")

✅ Silver layer saved successfully!
